## 1. Install dependencies

In [ ]:
!pip -q install kagglehub thop python-docx xgboost --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 7.5 MB/s eta 0:00:00


## 2. Download the dataset

In [ ]:
import kagglehub

path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")
print("Path to dataset files:", path)

import os
for root, dirs, files in os.walk(path):
    depth = root[len(path):].count(os.sep)
    if depth < 2:
        print(root, "->", dirs[:10])

Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Path to dataset files: /kaggle/input/skin-cancer9-classesisic
/kaggle/input/skin-cancer9-classesisic -> ['Skin cancer ISIC The International Skin Imaging Collaboration']
/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration -> ['Test', 'Train']


## 3. Config

Adjust `EPOCHS`, `BATCH_SIZE`, `IMG_SIZE` here if you need a faster/slower run.

In [ ]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10          # increase (e.g. 20-30) for better accuracy, decrease for a quick run
LR = 1e-4
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
SEED = 42
NUM_WORKERS = 2

torch.manual_seed(SEED)

Using device: cuda


## 4. Data loading

Handles both possible dataset layouts:
- `Train/<class>/*.jpg` + `Test/<class>/*.jpg`
- a single flat `<class>/*.jpg` folder (in which case we split it ourselves)

In [ ]:
import os, glob
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

def find_data_root(base_path):
    # Look for a level that contains class subfolders directly, or Train/Test dirs
    candidates = []
    for root, dirs, files in os.walk(base_path):
        lower_dirs = [d.lower() for d in dirs]
        if "train" in lower_dirs and "test" in lower_dirs:
            candidates.append(root)
    if candidates:
        return candidates[0], True
    return base_path, False

data_root, has_split = find_data_root(path)
print("Data root:", data_root, "| has Train/Test split:", has_split)

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

if has_split:
    train_dir = os.path.join(data_root, [d for d in os.listdir(data_root) if d.lower() == "train"][0])
    test_dir  = os.path.join(data_root, [d for d in os.listdir(data_root) if d.lower() == "test"][0])
    full_train = datasets.ImageFolder(train_dir, transform=train_tf)
    test_ds    = datasets.ImageFolder(test_dir, transform=eval_tf)
    n_val = int(len(full_train) * VAL_SPLIT)
    n_train = len(full_train) - n_val
    train_ds, val_ds = random_split(full_train, [n_train, n_val],
                                     generator=torch.Generator().manual_seed(SEED))
    class_names = full_train.classes
else:
    full_ds = datasets.ImageFolder(data_root, transform=eval_tf)
    n_total = len(full_ds)
    n_test = int(n_total * TEST_SPLIT)
    n_val = int(n_total * VAL_SPLIT)
    n_train = n_total - n_val - n_test
    train_ds, val_ds, test_ds = random_split(full_ds, [n_train, n_val, n_test],
                                              generator=torch.Generator().manual_seed(SEED))
    train_ds.dataset.transform = train_tf  # note: shares underlying dataset; fine for this quick split
    class_names = full_ds.classes

NUM_CLASSES = len(class_names)
print("Classes:", class_names)
print("Train / Val / Test sizes:", len(train_ds), len(val_ds), len(test_ds))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

Data root: /kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration | has Train/Test split: True
Classes: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']
Train / Val / Test sizes: 1904 335 118


## 5. Model zoo — build each backbone with a fresh classification head

In [ ]:
import torch.nn as nn
from torchvision import models

def build_model(name, num_classes):
    if name == "AlexNet":
        m = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
        feat_layer = "classifier.5"  # penultimate FC (after ReLU) — used later for feature extraction
    elif name == "VGG16":
        m = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
        feat_layer = "classifier.5"
    elif name == "VGG19":
        m = models.vgg19(weights=models.VGG19_Weights.DEFAULT)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
        feat_layer = "classifier.5"
    elif name == "ResNet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        feat_layer = "avgpool"
    elif name == "ResNet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        feat_layer = "avgpool"
    elif name == "ResNet101":
        m = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        feat_layer = "avgpool"
    elif name == "DenseNet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
        feat_layer = "features"
    elif name == "EfficientNet-B0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        feat_layer = "avgpool"
    else:
        raise ValueError(name)
    return m, feat_layer

MODEL_NAMES = ["AlexNet", "VGG16", "VGG19", "ResNet18", "ResNet50",
               "ResNet101", "DenseNet121", "EfficientNet-B0"]

## 6. Training + evaluation utilities

In [ ]:
import numpy as np
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import torch.nn.functional as F

def train_one_model(model, train_loader, val_loader, epochs, lr, device):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_state = None
    best_val_acc = -1
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            opt.step()
        # quick val check
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb).argmax(1)
                correct += (pred == yb).sum().item()
                total += yb.size(0)
        val_acc = correct / total
        print(f"  epoch {epoch+1}/{epochs}  val_acc={val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model

@torch.no_grad()
def evaluate(model, loader, device, num_classes):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    for xb, yb in loader:
        xb = xb.to(device)
        out = model(xb)
        probs = F.softmax(out, dim=1).cpu().numpy()
        preds = probs.argmax(1)
        all_probs.append(probs)
        all_preds.append(preds)
        all_labels.append(yb.numpy())
    all_probs = np.concatenate(all_probs)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    acc = accuracy_score(all_labels, all_preds) * 100
    prec = precision_score(all_labels, all_preds, average="macro", zero_division=0) * 100
    rec = recall_score(all_labels, all_preds, average="macro", zero_division=0) * 100
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0) * 100
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro") * 100
    except Exception as e:
        auc = float("nan")
    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1, "AUC": auc}, all_probs, all_labels

## 7. Table 1 — Train and evaluate all 8 backbones


In [ ]:
table1_rows = []
trained_models = {}   # keep trained weights so we can pick the best one for feature extraction

for name in MODEL_NAMES:
    print(f"=== Training {name} ===")
    model, feat_layer = build_model(name, NUM_CLASSES)
    model = train_one_model(model, train_loader, val_loader, EPOCHS, LR, DEVICE)
    metrics, _, _ = evaluate(model, test_loader, DEVICE, NUM_CLASSES)
    print(f"{name}: {metrics}")
    row = {"Model": name, **metrics}
    table1_rows.append(row)
    trained_models[name] = (model.cpu(), feat_layer)
    torch.cuda.empty_cache()

import pandas as pd
table1_df = pd.DataFrame(table1_rows)[["Model", "Accuracy", "Precision", "Recall", "F1-Score", "AUC"]]
table1_df = table1_df.round(2)
table1_df

=== Training AlexNet ===
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:02<00:00, 110MB/s]


  epoch 1/10  val_acc=0.6090
  epoch 2/10  val_acc=0.6090
  epoch 3/10  val_acc=0.7134
  epoch 4/10  val_acc=0.6836
  epoch 5/10  val_acc=0.6507
  epoch 6/10  val_acc=0.7075
  epoch 7/10  val_acc=0.6776
  epoch 8/10  val_acc=0.6985
  epoch 9/10  val_acc=0.6836
  epoch 10/10  val_acc=0.7194
AlexNet: {'Accuracy': 52.54237288135594, 'Precision': 48.009950088897455, 'Recall': 52.083333333333336, 'F1-Score': 47.3053008418862, 'AUC': np.float64(89.05098512835087)}
=== Training VGG16 ===
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:04<00:00, 125MB/s]


  epoch 1/10  val_acc=0.5373
  epoch 2/10  val_acc=0.6239
  epoch 3/10  val_acc=0.5552
  epoch 4/10  val_acc=0.6149
  epoch 5/10  val_acc=0.6716
  epoch 6/10  val_acc=0.6806
  epoch 7/10  val_acc=0.6388
  epoch 8/10  val_acc=0.7104
  epoch 9/10  val_acc=0.7045
  epoch 10/10  val_acc=0.6746
VGG16: {'Accuracy': 50.847457627118644, 'Precision': 51.342652175985506, 'Recall': 50.69444444444444, 'F1-Score': 44.99165995812234, 'AUC': np.float64(89.94417211328974)}
=== Training VGG19 ===
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


...

100%|██████████| 30.8M/30.8M [00:00<00:00, 177MB/s]


  epoch 1/10  val_acc=0.6269
  epoch 2/10  val_acc=0.7343
  epoch 3/10  val_acc=0.7403
  epoch 4/10  val_acc=0.7194
  epoch 5/10  val_acc=0.7552
  epoch 6/10  val_acc=0.7284
  epoch 7/10  val_acc=0.7701
  epoch 8/10  val_acc=0.7403
  epoch 9/10  val_acc=0.7433
  epoch 10/10  val_acc=0.7642
DenseNet121: {'Accuracy': 56.779661016949156, 'Precision': 58.70412746842953, 'Recall': 55.55555555555556, 'F1-Score': 51.48124834115172, 'AUC': np.float64(92.71881216254617)}
=== Training EfficientNet-B0 ===
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 142MB/s]


  epoch 1/10  val_acc=0.5761
  epoch 2/10  val_acc=0.6627
  epoch 3/10  val_acc=0.7254
  epoch 4/10  val_acc=0.7313
  epoch 5/10  val_acc=0.7612
  epoch 6/10  val_acc=0.7761
  epoch 7/10  val_acc=0.7642
  epoch 8/10  val_acc=0.7522
  epoch 9/10  val_acc=0.7731
  epoch 10/10  val_acc=0.7284
EfficientNet-B0: {'Accuracy': 56.779661016949156, 'Precision': 61.79948875601049, 'Recall': 55.55555555555556, 'F1-Score': 53.844367637471095, 'AUC': np.float64(91.61823908307285)}


,Model,Accuracy,Precision,Recall,F1-Score,AUC
0,AlexNet,52.54,48.01,52.08,47.31,89.05
1,VGG16,50.85,51.34,50.69,44.99,89.94
2,VGG19,52.54,61.35,52.08,49.92,89.30
3,ResNet18,57.63,58.57,56.25,54.23,90.94
4,ResNet50,53.39,45.90,52.78,46.61,90.93
5,ResNet101,60.17,62.66,58.33,55.35,93.06
6,DenseNet121,56.78,58.70,55.56,51.48,92.72
7,EfficientNet-B0,56.78,61.80,55.56,53.84,91.62


## 8. Table 2 — Deep-feature extraction + classical classifiers


In [ ]:
best_name = table1_df.sort_values("Accuracy", ascending=False).iloc[0]["Model"]
print("Best backbone for feature extraction:", best_name)
best_model, feat_layer = trained_models[best_name]
best_model = best_model.to(DEVICE).eval()

# Register a forward hook on the penultimate layer
activation = {}
def get_hook(name):
    def hook(module, inp, out):
        activation[name] = out.detach()
    return hook

layer = dict(best_model.named_modules())[feat_layer]
handle = layer.register_forward_hook(get_hook("feat"))

@torch.no_grad()
def extract_features(loader):
    feats, labels = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        _ = best_model(xb)
        f = activation["feat"]
        f = torch.flatten(f, 1).cpu().numpy()
        feats.append(f)
        labels.append(yb.numpy())
    return np.concatenate(feats), np.concatenate(labels)

X_train, y_train = extract_features(train_loader)
X_val, y_val = extract_features(val_loader)
X_test, y_test = extract_features(test_loader)
handle.remove()

# combine train+val for classical classifiers (they don't need a held-out val split the same way)
X_train_full = np.concatenate([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])
print("Feature dim:", X_train_full.shape[1])

Best backbone for feature extraction: ResNet101
Feature dim: 2048


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

scaler = StandardScaler().fit(X_train_full)
X_train_s = scaler.transform(X_train_full)
X_test_s = scaler.transform(X_test)

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=SEED),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5),
    "Linear SVM": SVC(kernel="linear", probability=True, random_state=SEED),
    "RBF-SVM": SVC(kernel="rbf", probability=True, random_state=SEED),
    "XGBoost": XGBClassifier(n_estimators=300, use_label_encoder=False,
                              eval_metric="mlogloss", random_state=SEED),
}

table2_rows = []
for cname, clf in classifiers.items():
    print(f"Training {cname} ...")
    clf.fit(X_train_s, y_train_full)
    preds = clf.predict(X_test_s)
    probs = clf.predict_proba(X_test_s)
    acc = accuracy_score(y_test, preds) * 100
    prec = precision_score(y_test, preds, average="macro", zero_division=0) * 100
    rec = recall_score(y_test, preds, average="macro", zero_division=0) * 100
    f1 = f1_score(y_test, preds, average="macro", zero_division=0) * 100
    try:
        auc = roc_auc_score(y_test, probs, multi_class="ovr", average="macro") * 100
    except Exception:
        auc = float("nan")
    table2_rows.append({"Feature Extractor": "Deep Features", "Classifier": cname,
                         "Accuracy": acc, "Precision": prec, "Recall": rec,
                         "F1-Score": f1, "AUC": auc})

table2_df = pd.DataFrame(table2_rows)
table2_df = table2_df.round(2)
table2_df

Training Logistic Regression ...
Training Decision Tree ...
Training Random Forest ...
Training K-Nearest Neighbors (KNN) ...
Training Linear SVM ...
Training RBF-SVM ...
Training XGBoost ...


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [12:56:46] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,Feature Extractor,Classifier,Accuracy,Precision,Recall,F1-Score,AUC
0,Deep Features,Logistic Regression,57.63,58.98,56.25,52.12,88.12
1,Deep Features,Decision Tree,54.24,52.88,53.47,49.44,73.82
2,Deep Features,Random Forest,60.17,62.21,58.33,55.24,94.03
3,Deep Features,K-Nearest Neighbors (KNN),61.86,61.38,59.72,57.54,85.70
4,Deep Features,Linear SVM,57.63,70.07,59.26,57.01,93.04
5,Deep Features,RBF-SVM,62.71,66.02,60.42,59.32,94.52
6,Deep Features,XGBoost,58.47,62.25,56.94,54.55,89.21


## 9. Table 3 — Computational efficiency



In [ ]:
import io
from thop import profile

EFF_MODEL_NAMES = ["AlexNet", "VGG16", "VGG19", "ResNet18", "ResNet50",
                    "DenseNet121", "EfficientNet-B0"]

def model_size_mb(model):
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return len(buf.getvalue()) / (1024 ** 2)

def measure_inference_time_ms(model, device, img_size=IMG_SIZE, n_warmup=10, n_runs=50):
    model = model.to(device).eval()
    dummy = torch.randn(1, 3, img_size, img_size).to(device)
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(dummy)
        if device.type == "cuda":
            torch.cuda.synchronize()
        start = time.time()
        for _ in range(n_runs):
            _ = model(dummy)
        if device.type == "cuda":
            torch.cuda.synchronize()
        end = time.time()
    return ((end - start) / n_runs) * 1000  # ms

table3_rows = []
for name in EFF_MODEL_NAMES:
    model, _ = trained_models[name]
    model = model.to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    size_mb = model_size_mb(model)
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    flops, _ = profile(model, inputs=(dummy,), verbose=False)
    gflops = flops / 1e9
    inf_time = measure_inference_time_ms(model, DEVICE)
    acc = table1_df.loc[table1_df["Model"] == name, "Accuracy"].values[0]

    table3_rows.append({
        "Model": name,
        "Parameters (M)": round(n_params, 2),
        "Model Size (MB)": round(size_mb, 2),
        "FLOPs (G)": round(gflops, 2),
        "Inference Time (ms)": round(inf_time, 3),
        "Accuracy (%)": acc,
    })
    model.cpu()
    torch.cuda.empty_cache()

table3_df = pd.DataFrame(table3_rows)
table3_df

,Model,Parameters (M),Model Size (MB),FLOPs (G),Inference Time (ms),Accuracy (%)
0,AlexNet,57.04,217.60,0.71,2.077,52.54
1,VGG16,134.30,512.31,15.47,9.929,50.85
2,VGG19,139.61,532.57,19.63,12.128,52.54
3,ResNet18,11.18,42.73,1.82,2.285,57.63
4,ResNet50,23.53,90.05,4.13,5.841,53.39
5,DenseNet121,6.96,27.14,2.90,14.963,56.78
6,EfficientNet-B0,4.02,15.62,0.41,8.123,56.78


## 10. Fill the tables into your Word document


In [ ]:
from google.colab import files
print("Upload Task_01.docx:")
uploaded = files.upload()
docx_path = list(uploaded.keys())[0]

Upload Task_01.docx:


Saving Task 01.docx to Task 01.docx
